# ICA review and subject analysis

I use this walkthrough to inspect the decomposition before deciding which components to remove. The data here are **DEMONSTRATION ONLY**. Run with the installed environment and use a fresh output directory when repeating the notebook.

In [ ]:
from pathlib import Path
import mne
from IPython.display import Image, display
from eeg_workflow.demo import demo_config, synthetic_raw
from eeg_workflow.ica import save_review
from eeg_workflow.workflow import prepare_run, finish_run, analyse_run

repo = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
run = repo / "derivatives" / "notebook-subject"
config = demo_config()
raw = synthetic_raw(97)
print(raw.info)
print(raw.annotations)

In [ ]:
suggestions = prepare_run(raw, config, run, "demo-notebook", demonstration_only=True,
                          synthetic_suggestions=True)
display(suggestions)
for name in ("components_1.png", "sources.png", "source_psd.png"):
    display(Image(filename=str(run / "inspection" / name)))

## Inspect before excluding

The synthetic suggestion compares components with the generated blink trace. I inspect the maps, traces and spectra together before recording a decision. The static preview shows the first 15 seconds; the interactive calls below allow inspection across the recording. For your own data, replace the input/configuration and omit the synthetic suggestion option.

In [ ]:
prepared = mne.io.read_raw_fif(run / "prepared-raw.fif", preload=True)
ica = mne.preprocessing.read_ica(run / "model-ica.fif")
# Uncomment in a local interactive session.
# ica.plot_components()
# ica.plot_sources(prepared)
# ica.plot_properties(prepared, picks=[0, 1])

## Record the decision

Replace the empty list with the zero-based indices you decide to remove. Keeping every component is also an explicit decision. This cell records the decision for this run; it does not infer exclusions from the suggestion table.

In [ ]:
rejected_components = []  # Edit after inspecting this run.
save_review(run, rejected_components, reviewer="notebook researcher",
            reason="Synthetic walkthrough: retained components for this example")
epochs = finish_run(run)
print(epochs)

## Spectra, features and time-frequency power

I average the selected epochs within each condition, then export PSD curves, band features and Morlet power. Inspect the cleaned signal alongside these summaries. Every output in this walkthrough remains labelled **DEMONSTRATION ONLY**.

In [ ]:
features = analyse_run(run)
display(features.head())
for name in ("ica_comparison.png", "psd.png", "tfr.png"):
    display(Image(filename=str(run / name)))
print("Saved subject outputs:", run)